# Stage 3 — AlphaGo Zero

This notebook produces every number and plot of `reports/stage3.pdf` (specification, Section 7.4). It runs from top to bottom in a fresh Colab runtime and, unchanged, in Jupyter on a local machine. Its cells only import `az5`, call its functions, print tables, and draw plots; a cell longer than about 15 lines indicates logic that belongs in the package.

In [ ]:
# Setup: makes the package importable. In Colab it clones (or updates) your
# repository (a GitHub token with read access in the Colab secret
# GITHUB_TOKEN); on a local machine it switches to the repository root.
REPO_URL = "github.com/<user>/<repository>.git"    # edit once
import os, subprocess, sys
if "google.colab" in sys.modules:
    from google.colab import userdata
    if not os.path.exists("/content/repo"):
        url = f"https://{userdata.get('GITHUB_TOKEN')}@{REPO_URL}"
        subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run(["git", "pull", "-q"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
# Working directories of the training runs: on Google Drive in Colab, so that
# a run survives disconnects; in work/ (not committed) on a local machine.
RETRAIN = False        # True: run or resume training; False: use the weights in runs/
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/az5/main"
    ABLATION_WORKDIR = "/content/drive/MyDrive/az5/ablation"
else:
    WORKDIR = "work/main"
    ABLATION_WORKDIR = "work/ablation"

In [ ]:
%load_ext autoreload
%autoreload 2
from dataclasses import replace
import numpy as np
import matplotlib.pyplot as plt
import torch
from az5.config import Config
from az5.go import GoState
from az5.mcts import RolloutEvaluator
from az5.net import NetEvaluator, PolicyValueNet
from az5 import agents, arena, train

cfg = Config()
rng = np.random.default_rng(cfg.seed)

## Training

`train.run` writes a checkpoint after every iteration and resumes from it, so after a disconnect this cell is run again in a new session. When the run is complete, copy `log.csv` and the final `best_XXX.pt` from the working directory to `runs/log.csv` and `runs/best.pt` and commit them.

In [ ]:
if RETRAIN:
    train.run(cfg, WORKDIR)

In [ ]:
net = PolicyValueNet(cfg)
net.load_state_dict(torch.load("runs/best.pt"))
stage3_agent = agents.MCTSAgent(NetEvaluator(net), cfg.eval_simulations, "puct", cfg, rng)
stage2_agent = agents.MCTSAgent(RolloutEvaluator(rng), cfg.baseline_simulations, "uct", cfg, rng)

## (a) Public tests

In [ ]:
!python -m pytest -q

## (b) Learning curves

From `runs/log.csv`: value loss, policy loss, gate win rate with the accepted iterations marked, and anchor win rate with its interval, each against the iteration.

In [ ]:
# your calls to az5 here

## (c) Final match (completion criterion)

The Stage 3 agent against the Stage 2 agent, 50 pairs: win rate with its interval, win rates as Black and as White, mean time per move of both agents.

In [ ]:
# your calls to az5 here

## (d) Play-out equivalence

The Stage 3 agent against MCTS agents with 400 and 1 600 random play-outs, 25 pairs each.

In [ ]:
# your calls to az5 here

## (e) Value of search

The `PolicyAgent` of the final network against the Stage 2 agent and against the Stage 3 agent, 25 pairs each.

In [ ]:
# your calls to az5 here

## (f) Learned opening

The priors of the final network for the 25 points of the empty board as a 5×5 heat map, the prior of the pass, and the value of the empty board.

In [ ]:
# your calls to az5 here

## (g) Ablation

A second run of `cfg.iterations // 3` iterations in `ABLATION_WORKDIR` that differs from the main run in exactly one setting, for example `replace(cfg, iterations=cfg.iterations // 3, use_gate=False)`. Its final best network plays the main-run network of the same iteration, 25 pairs.

In [ ]:
# your calls to az5 here

## Playing against the agent

In [ ]:
# arena.play_game(agents.HumanAgent(), stage3_agent)    # you play Black